first pattern interrupt before

In [1]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated

In [2]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def draft_email(state: State) -> dict:
    """Draft an email — this runs automatically."""
    return {"messages": ["DRAFT: Dear Boss, I'd like to request a raise based on my performance..."]}

def send_email(state: State) -> dict:
    """Send the email — we want human approval first!"""
    return {"messages": ["✅ Email sent successfully!"]}

In [3]:
builder = StateGraph(State)
builder.add_node("draft", draft_email)
builder.add_node("send", send_email)
builder.add_edge(START, "draft")
builder.add_edge("draft", "send")
builder.add_edge("send", END)

In [4]:
memory = InMemorySaver()

In [5]:
memory = InMemorySaver()

# Compile with interrupt_before — graph pauses BEFORE "send" node runs
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["send"])           

config = {"configurable": {"thread_id": "email_1"}}

In [6]:
result = graph.invoke(
    {"messages": [HumanMessage(content="Write an email asking for a raise")]},
    config=config
)

print("📝 Draft:", result["messages"][-1].content)
print("\nDo you approve sending this? (y/n)")
approval = input("> ")

if approval.lower() == "y":
    result = graph.invoke(None, config=config)
    print(result["messages"][-1].content)   
else:
    print("❌ Email cancelled by human.")

📝 Draft: DRAFT: Dear Boss, I'd like to request a raise based on my performance...

Do you approve sending this? (y/n)


✅ Email sent successfully!


Second pattern interrupt after

In [7]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated

In [8]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def draft_email(state: State) -> dict:
    """Draft an email — this runs automatically."""
    return {"messages": ["DRAFT: Dear Boss, I'd like to request a raise based on my performance..."]}

def send_email(state: State) -> dict:
    """Send the email — we want human approval first!"""
    return {"messages": ["✅ Email sent successfully!"]}

In [9]:
builder = StateGraph(State)
builder.add_node("draft", draft_email)
builder.add_node("send", send_email)
builder.add_edge(START, "draft")
builder.add_edge("draft", "send")
builder.add_edge("send", END)

In [10]:
memory = InMemorySaver()

In [11]:
memory = InMemorySaver()

# Compile with interrupt_after — graph pauses AFTER "send" node runs
graph = builder.compile(
    checkpointer=memory,
    interrupt_after=["draft"])           

config = {"configurable": {"thread_id": "email_1"}}


In [14]:
result = graph.invoke(
    {"messages": [HumanMessage(content="Write an email asking for a raise")]},
    config=config
)

print("📝 Draft:", result["messages"][-1].content)
print("\nDo you approve sending this? (y/n)")
approval = input("> ")

if approval.lower() == "y":
    result = graph.invoke(None, config=config)
    print(result["messages"][-1].content)   
else:
    print("❌ Email cancelled by human.")

📝 Draft: DRAFT: Dear Boss, I'd like to request a raise based on my performance...

Do you approve sending this? (y/n)
❌ Email cancelled by human.


Third pattern interrupt function

In [15]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import interrupt, Command
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated

In [16]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def draft_email(state: State) -> dict:
    return {"messages": ["DRAFT: Dear Boss, I'd like to request a raise based on my performance..."]}

def review_email(state: State) -> dict:
    response = interrupt({"question": "Approve this email?", "options": "yes / no"})
    if response.lower() in ["yes", "y"]:
        return {"messages": ["✅ Email sent successfully!"]}
    return {"messages": [f"❌ Email cancelled. Reason: {response}"]}

In [17]:
builder = StateGraph(State)
builder.add_node("draft", draft_email)
builder.add_node("review", review_email)
builder.add_edge(START, "draft")
builder.add_edge("draft", "review")
builder.add_edge("review", END)
graph = builder.compile(checkpointer=InMemorySaver())

In [20]:
config = {"configurable": {"thread_id": "email_1"}}

result = graph.invoke({"messages": [HumanMessage(content="Write a raise email")]}, config)
print("📝 Draft:", result["messages"][-1].content)

approval = input("Approve? (yes/no): ")
result = graph.invoke(Command(resume=approval), config)
print(result["messages"][-1].content)

📝 Draft: DRAFT: Dear Boss, I'd like to request a raise based on my performance...
❌ Email cancelled. Reason: no


Update State; Human in the Loop Pattern

In [21]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated

In [22]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

def draft_email(state: State) -> dict:
    return {"messages": ["DRAFT: Dear Boss, I'd like to request a raise based on my performance..."]}

def send_email(state: State) -> dict:
    return {"messages": ["✅ Email sent successfully!"]}

In [23]:
builder = StateGraph(State)
builder.add_node("draft", draft_email)
builder.add_node("send", send_email)
builder.add_edge(START, "draft")
builder.add_edge("draft", "send")
builder.add_edge("send", END)

In [24]:
graph = builder.compile(checkpointer=InMemorySaver(), interrupt_before=["send"])
config = {"configurable": {"thread_id": "email_1"}}

result = graph.invoke({"messages": [HumanMessage(content="Write a raise email")]}, config)
print("📝 Draft:", result["messages"][-1].content)

📝 Draft: DRAFT: Dear Boss, I'd like to request a raise based on my performance...


In [26]:
approval = input("Approve? Edit? (yes/edit/no): ")

if approval.lower() == "edit":
    # Inject a new message into the state before resuming
    graph.update_state(config, {
        "messages": [HumanMessage(content="Make it more polite and mention 3 years of service")]
    })
    result = graph.invoke(None, config)
    print(result["messages"][-1].content)

elif approval.lower() in ["yes", "y"]:
    result = graph.invoke(None, config)
    print(result["messages"][-1].content)

else:
    print("❌ Email cancelled.")

✅ Email sent successfully!
